In [3]:
import pandas as pd
import numpy as np
from scipy.stats import skew
from sklearn.preprocessing import RobustScaler

# 1. 讀取資料
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

# ==========================================
# 2. 移除離群值 (必須在拆分特徵與目標變數之前做！)
# ==========================================
outliers = train[(train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)].index
train = train.drop(outliers)

# 3. 目標變數轉換 (對數轉換以處理偏態)
y_train = np.log1p(train["SalePrice"])

# 將 train 與 test 合併進行特徵工程 (捨棄不必要的 Id 欄位)
train_features = train.drop(["SalePrice", "Id"], axis=1)
test_features = test.drop(["Id"], axis=1)
all_data = pd.concat([train_features, test_features]).reset_index(drop=True)

# ==========================================
# 4. 數值型態修正 (將「數字代表類別」的變數轉為字串)
# ==========================================
for col in ['MSSubClass', 'MoSold', 'YrSold']:
    all_data[col] = all_data[col].apply(str)

# 5. 缺失值填補
# 依據同一個 Neighborhood 的 LotFrontage 中位數填補
all_data["LotFrontage"] = all_data.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median())
)

numeric_cols = all_data.select_dtypes(exclude=['object']).columns
categorical_cols = all_data.select_dtypes(include=['object']).columns

# 數值特徵填補 0，類別特徵填補 None
all_data[numeric_cols] = all_data[numeric_cols].fillna(0)
all_data[categorical_cols] = all_data[categorical_cols].fillna("None")

# ==========================================
# 6. 進階特徵工程
# ==========================================
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['HouseAge'] = all_data['YrSold'].astype(int) - all_data['YearBuilt']
all_data['RemodelAge'] = all_data['YrSold'].astype(int) - all_data['YearRemodAdd']
all_data['TotalBath'] = all_data['FullBath'] + (0.5 * all_data['HalfBath']) + \
                        all_data['BsmtFullBath'] + (0.5 * all_data['BsmtHalfBath'])

# 二元特徵標籤
all_data['HasPool'] = all_data['PoolArea'].apply(lambda x: 1 if x > 0 else 0)
all_data['Has2ndFloor'] = all_data['2ndFlrSF'].apply(lambda x: 1 if x > 0 else 0)
all_data['HasGarage'] = all_data['GarageArea'].apply(lambda x: 1 if x > 0 else 0)
all_data['HasBsmt'] = all_data['TotalBsmtSF'].apply(lambda x: 1 if x > 0 else 0)
all_data['HasFireplace'] = all_data['Fireplaces'].apply(lambda x: 1 if x > 0 else 0)

# ==========================================
# 7. 順序型特徵轉換 (Ordinal Encoding) - 讓 XGBoost 變精準的關鍵！
# ==========================================
qual_mapping = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
             'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond']

for col in qual_cols:
    all_data[col] = all_data[col].map(qual_mapping)

# ==========================================
# 8. 處理數值特徵的偏態 (Skewness) - 讓線性模型變準的關鍵！
# ==========================================
# 重新抓取更新後的數值欄位
numeric_cols = all_data.select_dtypes(exclude=['object']).columns
skewed_feats = all_data[numeric_cols].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)

# 找出偏度大於 0.75 的特徵並做 Log 轉換
high_skew = skewed_feats[skewed_feats > 0.75].index
all_data[high_skew] = np.log1p(all_data[high_skew])

# ==========================================
# 9. 類別特徵 One-Hot Encoding
# ==========================================
all_data = pd.get_dummies(all_data)
print(f"特徵工程與 Encoding 後的資料維度: {all_data.shape}")

# 10. 將資料切分回 train 與 test
X_train = all_data.iloc[:len(y_train), :]
X_test = all_data.iloc[len(y_train):, :]

# ==========================================
# 11. 特徵縮放 (RobustScaler 抵抗潛在極端值)
# ==========================================
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 將 numpy array 轉回 DataFrame 以保留欄位名稱
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print(f"處理完畢！X_train 維度: {X_train.shape}, X_test 維度: {X_test.shape}")

C:\Users\austi\AppData\Local\Temp\ipykernel_17952\4041742430.py:37: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = all_data.select_dtypes(include=['object']).columns


特徵工程與 Encoding 後的資料維度: (2917, 310)
處理完畢！X_train 維度: (1458, 310), X_test 維度: (1459, 310)


In [4]:
X_train.to_csv(
    "../data/X_train.csv",
    index=False
)

X_test.to_csv(
    "../data/X_test.csv",
    index=False
)

pd.DataFrame(y_train).to_csv(
    "../data/y_train.csv",
    index=False
)

print("資料已儲存")

資料已儲存
